# PDB Graph Store usage exemple

## Installing dependences

In [1]:
!sudo apt update > /dev/null; sudo apt install -y gcc > /dev/null

In [ ]:
!pip install -r ./../r.txt > /dev/null

## Importing graphein modules

In [1]:
from graphein.protein.config import ProteinGraphConfig
from graphein.protein.graphs import construct_graph
from graphein.protein.utils import download_pdb

## Importing graph store modules

In [2]:
import Builder
from PDBGraphStore import PDBGraphStore

In [3]:
from operations import remove_graph_from_store
from operations import split_graph_store
from operations import merge_graph_stores

In [4]:
import edge_functions_Model as edgeModel

## Importing system modules

In [5]:
import traceback
import os

## Reading dataset toy.txt at ./../data/

In [7]:
def read_dataset(general_data_path, dataset_txt_name, file_mode='r'):
    pdb_codes = list()
    with open(f"{general_data_path}/{dataset_txt_name}", file_mode) as file:
        for line in file:
            if line[0] != '#':
                pdb_codes.append(line.strip().upper())

    return pdb_codes

pdb_codes = read_dataset("./../data", "toy.txt", 'r')
pdb_codes

['1BXL',
 '1G5J',
 '1TY4',
 '1ZY3',
 '2A5Y',
 '2BZW',
 '2JM6',
 '2K7W',
 '2KBW',
 '2LP8',
 '2LR1',
 '2M04',
 '2M5B',
 '2MEJ',
 '2NL9']

## Defining the edge functions

In [8]:
def define_graphein_edge_funcs():
   return [edgeModel.edge_functions_dict[f] for f in ["delaunay", "aromatic", "aromatic_sulphur"]]

edge_funcs = define_graphein_edge_funcs()

edge_funcs

[<function graphein.protein.edges.distance.add_delaunay_triangulation(G: 'nx.Graph', allowable_nodes: 'Optional[List[str]]' = None)>,
 <function graphein.protein.edges.distance.add_aromatic_interactions(G: 'nx.Graph', pdb_df: 'Optional[pd.DataFrame]' = None)>,
 <function graphein.protein.edges.distance.add_aromatic_sulphur_interactions(G: 'nx.Graph', rgroup_df: 'Optional[pd.DataFrame]' = None)>]

## Defining graph configuration (with edge functions and node granularity)

In [9]:
def define_configuration(edge_construction_funcs):
    return {
        "granularity": "CA",
        "edge_construction_functions": edge_construction_funcs
    }

graph_config = ProteinGraphConfig(**define_configuration(edge_funcs))

graph_config

ProteinGraphConfig(granularity='CA', keep_hets=[], insertions=True, alt_locs='max_occupancy', pdb_dir=None, verbose=False, exclude_waters=True, deprotonate=False, protein_df_processing_functions=None, edge_construction_functions=[<function add_delaunay_triangulation at 0x7f4be7f56160>, <function add_aromatic_interactions at 0x7f4be7f55b20>, <function add_aromatic_sulphur_interactions at 0x7f4be7f55bc0>], node_metadata_functions=[<function meiler_embedding at 0x7f4be7f64720>], edge_metadata_functions=None, graph_metadata_functions=None, get_contacts_config=None, dssp_config=None)

In [10]:
def get_pdb_file(pdb_data_path, pdb_code):
    pdb_code = pdb_code.lower()
    if os.path.exists(f"{pdb_data_path}/{pdb_code}.pdb"):
        print(f"Reading {pdb_code} from local directory")
        pdb_file = os.path.abspath(f"{pdb_data_path}/{pdb_code}.pdb")
    else:
        print(f"Downloading {pdb_code} from PDB")
        try:
            pdb_file = download_pdb(pdb_code, f"{pdb_data_path}/")
        except Exception as e:
            raise e

    if pdb_file == None:
        raise Exception("Error reading the pdb file")

    return pdb_file

In [11]:
def prepare_graph(pdb_data_path, pdb_code):
    protein_graphs = dict()

    try:
        pdb_file = get_pdb_file(pdb_data_path=pdb_data_path, pdb_code=pdb_code)
    except Exception as e:
        raise e

    graph = construct_graph(config=graph_config, path=pdb_file)
    graph.graph["pdb_code"] = pdb_code
    print(graph, "\n")

    config = graph.graph["config"]
    graph_pdb_code = graph.graph["pdb_code"]
    graph.graph.clear()
    graph.graph["config"] = config
    graph.graph["pdb_code"] = graph_pdb_code

    protein_graphs.setdefault(pdb_code, []).append(graph.copy())

    return protein_graphs

In [12]:
def build_pdb_store():
    protein_graphs = {}

    for _, pdb_code in enumerate(pdb_codes.copy()):
        try:
            graph = prepare_graph("./../data/pdb_files", pdb_code)
        except Exception as e:
            msg = traceback.format_exc()
            print(msg)
            continue
        protein_graphs[pdb_code] = graph[pdb_code]

    body_parts, _ = Builder.compress_pdb_graphs(protein_graphs)
    pdb_store = PDBGraphStore(body_parts)

    return pdb_store

In [13]:
pdb_graph_store = build_pdb_store()
pdb_graph_store

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Reading 1bxl from local directory


...

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/conda/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Graph named '2mej' with 339 nodes and 2454 edges 

Reading 2nl9 from local directory


Graph named '2nl9' with 163 nodes and 1120 edges 

number of edges: 24911
dict_keys(['pdb_code_to_id', 'pdb_id_to_nodes', 'pdb_id_to_edges', 'node_label_to_node_id', 'edge_label_to_edge_id', 'edge_attr_keys', 'node_global_attr_keys', 'node_local_attr_keys', 'node_attr_values', 'edge_attr_values', 'node_global_attr_keyvalue_mapping', 'node_local_attr_keyvalue_mapping', 'edge_local_attr_keyvalue_mapping'])


## Playing with pdb graph store object

In [14]:
print(pdb_graph_store)

PDBGraphStore with 15 pdbs


#### Getting the list of pdb codes in this object

In [15]:
print(pdb_graph_store.get_this_pdb_list())

dict_keys(['1BXL', '1G5J', '1TY4', '1ZY3', '2A5Y', '2BZW', '2JM6', '2K7W', '2KBW', '2LP8', '2LR1', '2M04', '2M5B', '2MEJ', '2NL9'])


#### Extracting one pdb

In [16]:
g = pdb_graph_store.extract_pdb('1BXL')

print(g)

Graph with 197 nodes and 1384 edges


#### Removing one pdb

In [17]:
pdb_graph_store = remove_graph_from_store(['1BXL'], pdb_graph_store)

print(pdb_graph_store)

KeyboardInterrupt: 

In [ ]:
print(pdb_graph_store.get_this_pdb_list())

#### Inserting one pdb

In [ ]:
pdb_graph_store.insert_pdb({"1BXL": [g]})

print(pdb_graph_store)

In [ ]:
print(pdb_graph_store.get_this_pdb_list())

#### Spliting pdb store into 2 distincts stores

In [ ]:
store1, store2 = split_graph_store(pdb_graph_store, ["1BXL", "1G5J"])

In [ ]:
print(store1.get_this_pdb_list())
print(store2.get_this_pdb_list())

In [ ]:
merged = merge_graph_stores([store1, store2])

print(merged.get_this_pdb_list())